In [1]:
"""Tokenize train/{lang}.txt -> bin/{tokname}/{lang}.bin (uint16).
Then mix() interleaves them at a target token ratio.
"""
import os, time
import numpy as np
import tiktoken
import shutil, pickle

ROOT = "D:/Tasks/Project_SLM"
EOT_NAME = "<|endoftext|>"
BATCH = 2048                       # docs per encode_ordinary_batch call
TOTAL_TOKENS = 9_000_000_000       # training budget
VAL_TOKENS = 15_000_000
RATIO = (("en", .40), ("de", .35), ("te", .25))
VOCAB_PADDED = 48064               # 48001 rounded up to a multiple of 64       # docs per encode_ordinary_batch call



In [2]:
def load_tok(name):
    """Rebuild vocab from a minbpe .model, return a tiktoken Encoding."""
    path = f"{ROOT}/tok/{name}.model"
    vocab = {i: bytes([i]) for i in range(256)}
    with open(path, encoding="utf-8") as f:
        assert f.readline().strip() == "minbpe v1"
        pat = f.readline().rstrip("\n")
        for _ in range(int(f.readline())):
            f.readline()
        nid = 256
        for line in f:
            a, b = map(int, line.split())
            vocab[nid] = vocab[a] + vocab[b]
            nid += 1
    ranks = {v: k for k, v in vocab.items()}
    assert len(ranks) == len(vocab), "duplicate token bytes - vocab corrupt"
    eot = len(vocab)
    assert eot < VOCAB_PADDED, f"eot {eot} >= padded vocab {VOCAB_PADDED}"
    enc = tiktoken.Encoding(name=name, pat_str=pat, mergeable_ranks=ranks,
                            special_tokens={EOT_NAME: eot})
    return enc, eot
 
 
def _flush(enc, eot, batch, fo):
    ids = enc.encode_ordinary_batch(batch, num_threads=8)
    flat = []
    for seq in ids:
        flat.extend(seq); flat.append(eot)
    arr = np.array(flat, dtype=np.uint32)
    assert arr.max() < 65536, f"token id {arr.max()} overflows uint16"
    arr.astype(np.uint16).tofile(fo)
    return len(flat)
 
 
def tokenize(tokname, lang, split="train"):
    enc, eot = load_tok(tokname)
    out_dir = f"{ROOT}/bin/{tokname}"
    os.makedirs(out_dir, exist_ok=True)
    suffix = "" if split == "train" else "_val"
    out = f"{out_dir}/{lang}{suffix}.bin"
    n_tok, n_doc, t0 = 0, 0, time.time()
    with open(f"{ROOT}/{split}/{lang}.txt", encoding="utf-8", errors="replace") as f, \
         open(out, "wb") as fo:
        batch = []
        for line in f:
            batch.append(line.rstrip("\n"))
            if len(batch) >= BATCH:
                n_tok += _flush(enc, eot, batch, fo)
                n_doc += len(batch); batch = []
                if n_doc % 200_000 < BATCH:
                    print(f"    {n_doc:,} docs  {n_tok/1e6:.0f}M tok  "
                          f"{time.time()-t0:.0f}s", flush=True)
        if batch:
            n_tok += _flush(enc, eot, batch, fo); n_doc += len(batch)
    print(f"  {tokname}/{lang}{suffix}: {n_doc:,} docs  {n_tok/1e6:.1f}M tokens  "
          f"{os.path.getsize(out)/1e9:.2f} GB  {time.time()-t0:.0f}s", flush=True)
    return n_tok
 
 
def mix(tokname, total_tokens=TOTAL_TOKENS, block=1024):
    """Interleave per-language bins into one shuffled training stream."""
    d = f"{ROOT}/bin/{tokname}"
    mm = {l: np.memmap(f"{d}/{l}.bin", dtype=np.uint16, mode="r") for l, _ in RATIO}
    want = {l: int(total_tokens * p) for l, p in RATIO}
    bad = False
    for l, w in want.items():
        ep = w / len(mm[l])
        flag = "  <-- OVER 1 EPOCH" if ep > 1.0 else ""
        print(f"  {l}: want {w/1e9:.2f}B  have {len(mm[l])/1e9:.2f}B  "
              f"epochs {ep:.2f}{flag}")
        bad |= ep > 1.0
    if bad:
        print("  WARNING: repetition asymmetry between languages. "
              "Lower total_tokens.")
    rng = np.random.default_rng(0)
    langs = [l for l, _ in RATIO]
    order = np.concatenate([np.full(want[l] // block, i) for i, l in enumerate(langs)])
    rng.shuffle(order)
    pos = {l: 0 for l in langs}
    with open(f"{d}/train.bin", "wb") as fo:
        for i in order:
            l = langs[i]
            if pos[l] + block > len(mm[l]):
                pos[l] = 0
            mm[l][pos[l]:pos[l] + block].tofile(fo)
            pos[l] += block
    print(f"  -> {d}/train.bin  {os.path.getsize(d+'/train.bin')/1e9:.1f} GB")
 
 
def mix_val(tokname, total_tokens=VAL_TOKENS):
    d = f"{ROOT}/bin/{tokname}"
    with open(f"{d}/val.bin", "wb") as fo:
        for l, p in RATIO:
            a = np.memmap(f"{d}/{l}_val.bin", dtype=np.uint16, mode="r")
            n = min(int(total_tokens * p), len(a))
            a[:n].tofile(fo)
    print(f"  -> {d}/val.bin  {os.path.getsize(d+'/val.bin')/1e6:.0f} MB")
 
 
def write_meta(tokname):
    with open(f"{ROOT}/bin/{tokname}/meta.pkl", "wb") as f:
        pickle.dump({"vocab_size": VOCAB_PADDED}, f)
 
 
def free_gb(path=ROOT):
    return shutil.disk_usage(os.path.splitdrive(os.path.abspath(path))[0] or "/").free / 1e9
 
 

In [3]:
print(f"free space: {free_gb():.1f} GB")
for tokname in ("o200k_pat", "cl100k_pat"):      # o200k first — the one you need
    print(f"\n=== {tokname} ===", flush=True)
    for lang in ("te", "de", "en"):
        tokenize(tokname, lang, "train")
        tokenize(tokname, lang, "heldout")
    mix(tokname)
    mix_val(tokname)
    write_meta(tokname)
    # per-language bins are no longer needed once train.bin/val.bin exist
    for lang in ("te", "de", "en"):
        for sfx in ("", "_val"):
            p = f"{ROOT}/bin/{tokname}/{lang}{sfx}.bin"
            if os.path.exists(p):
                os.remove(p)
    print(f"  cleaned intermediates, free space: {free_gb():.1f} GB")
    

free space: 94.8 GB

=== o200k_pat ===
    200,704 docs  167M tok  53s
    401,408 docs  319M tok  100s
    600,064 docs  463M tok  155s
    800,768 docs  605M tok  213s
    1,001,472 docs  741M tok  270s
    1,200,128 docs  874M tok  327s
    1,400,832 docs  1006M tok  384s
    1,601,536 docs  1137M tok  439s
    1,800,192 docs  1265M tok  494s
    2,000,896 docs  1374M tok  542s
    2,201,600 docs  1474M tok  588s
    2,400,256 docs  1574M tok  634s
    2,600,960 docs  1674M tok  681s
    2,801,664 docs  1775M tok  728s
    3,000,320 docs  1874M tok  774s
    3,201,024 docs  1974M tok  820s
    3,401,728 docs  2075M tok  864s
    3,600,384 docs  2174M tok  910s
    3,801,088 docs  2274M tok  955s
    4,001,792 docs  2375M tok  1001s
  o200k_pat/te: 4,001,909 docs  2375.2M tokens  4.75 GB  1001s
  o200k_pat/te_val: 5,774 docs  4.8M tokens  0.01 GB  2s
    200,704 docs  137M tok  50s
    401,408 docs  254M tok  95s
    600,064 docs  374M tok  140s
    800,768 docs  501M tok  188s
    1

In [3]:
import numpy as np
a = np.memmap("E:/Projects/Project_SLM/nanoGPT/data/o200k_pat/train.bin", dtype=np.uint16, mode="r")
b = np.memmap("E:/Projects/Project_SLM/nanoGPT/data/cl100k_pat/train.bin", dtype=np.uint16, mode="r")
print(len(a), len(b))
print("first 20 o200k:", a[:20].tolist())
print("first 20 cl100k:", b[:20].tolist())
print("identical:", np.array_equal(a[:100000], b[:100000]))

8999998464 8999998464
first 20 o200k: [84, 1744, 6415, 3839, 32292, 3534, 31471, 66, 2559, 39853, 31822, 90, 383, 2598, 8024, 25437, 58, 9354, 32, 1570]
first 20 cl100k: [84, 1654, 5679, 3483, 26822, 3222, 26164, 66, 2381, 32800, 26434, 90, 386, 2415, 7030, 21324, 58, 8156, 32, 1491]
identical: False


In [5]:
import sys; sys.path.insert(0, ".")
# from tokenize_corpus_final import load_tok
for name in ("o200k_pat", "cl100k_pat"):
    enc, _ = load_tok(name)
    arr = np.memmap(f"D:/Tasks/Project_SLM/bin/{name}/train.bin", dtype=np.uint16, mode="r")
    print(name, repr(enc.decode(arr[:60].astype(int).tolist()))[:150])

o200k_pat 'Taking Play Seriously By ROBIN MARANTZ HENIG Published: February 17, 2008 On a drizzly Tuesday night in late January, 200 people came out to hear a p
cl100k_pat 'Taking Play Seriously By ROBIN MARANTZ HENIG Published: February 17, 2008 On a drizzly Tuesday night in late January, 200 people came out to hear a p


In [ ]:

def alt_func():
    def load_tok(name):
        """Rebuild ranks+vocab from a minbpe .model, return a tiktoken Encoding."""
        path = f"{ROOT}/tok/{name}.model"
        vocab = {i: bytes([i]) for i in range(256)}
        with open(path, encoding="utf-8") as f:
            assert f.readline().strip() == "minbpe v1"
            pat = f.readline().rstrip("\n")
            for _ in range(int(f.readline())):
                f.readline()
            nid = 256
            for line in f:
                a, b = map(int, line.split())
                vocab[nid] = vocab[a] + vocab[b]
                nid += 1
        ranks = {v: k for k, v in vocab.items()}
        assert len(ranks) == len(vocab), "duplicate token bytes — vocab is corrupt"
        eot = len(vocab)
        enc = tiktoken.Encoding(name=name, pat_str=pat, mergeable_ranks=ranks,
                                special_tokens={EOT_NAME: eot})
        return enc, eot


    def tokenize(tokname, lang, chunk_docs=200_000):
        enc, eot = load_tok(tokname)
        out_dir = f"{ROOT}/bin/{tokname}"
        os.makedirs(out_dir, exist_ok=True)
        out = f"{out_dir}/{lang}.bin"
        n_tok, n_doc, t0 = 0, 0, time.time()
        with open(f"{ROOT}/train/{lang}.txt", encoding="utf-8") as f, \
            open(out, "wb") as fo:
            batch = []
            for line in f:
                batch.append(line.rstrip("\n"))
                if len(batch) >= BATCH:
                    n_tok += _flush(enc, eot, batch, fo)
                    n_doc += len(batch); batch = []
                    if n_doc % chunk_docs < BATCH:
                        print(f"    {n_doc:,} docs  {n_tok/1e6:.0f}M tok  "
                            f"{time.time()-t0:.0f}s", flush=True)
            if batch:
                n_tok += _flush(enc, eot, batch, fo); n_doc += len(batch)
        print(f"  {tokname}/{lang}: {n_doc:,} docs  {n_tok/1e6:.1f}M tokens  "
            f"{os.path.getsize(out)/1e9:.2f} GB  {time.time()-t0:.0f}s")
        return n_tok


    def tokenize(tokname, lang, split="train", chunk_docs=200_000):
        enc, eot = load_tok(tokname)
        out_dir = f"{ROOT}/bin/{tokname}"
        os.makedirs(out_dir, exist_ok=True)
        suffix = "" if split == "train" else "_val"
        out = f"{out_dir}/{lang}{suffix}.bin"
        n_tok, n_doc, t0 = 0, 0, time.time()
        with open(f"{ROOT}/{split}/{lang}.txt", encoding="utf-8") as f, \
            open(out, "wb") as fo:
            batch = []
            for line in f:
                batch.append(line.rstrip("\n"))
                if len(batch) >= BATCH:
                    n_tok += _flush(enc, eot, batch, fo)
                    n_doc += len(batch); batch = []
                    if n_doc % chunk_docs < BATCH:
                        print(f"    {n_doc:,} docs  {n_tok/1e6:.0f}M tok  "
                            f"{time.time()-t0:.0f}s", flush=True)
            if batch:
                n_tok += _flush(enc, eot, batch, fo); n_doc += len(batch)
        print(f"  {tokname}/{lang}: {n_doc:,} docs  {n_tok/1e6:.1f}M tokens  "
            f"{os.path.getsize(out)/1e9:.2f} GB  {time.time()-t0:.0f}s")
        return n_tok

    def _flush(enc, eot, batch, fo):
        ids = enc.encode_ordinary_batch(batch, num_threads=8)
        flat = []
        for seq in ids:
            flat.extend(seq); flat.append(eot)
        arr = np.array(flat, dtype=np.uint32)
        assert arr.max() < 65536, f"token id {arr.max()} overflows uint16"
        arr.astype(np.uint16).tofile(fo)
        return len(flat)


    def mix(tokname, ratio=(("en", .40), ("de", .35), ("te", .25)),
            total_tokens=3_000_000_000, block=1024): #8_000_000_000
        """Interleave per-language bins into one shuffled training stream."""
        d = f"{ROOT}/bin/{tokname}"
        mm = {l: np.memmap(f"{d}/{l}.bin", dtype=np.uint16, mode="r") for l, _ in ratio}
        want = {l: int(total_tokens * p) for l, p in ratio}
        for l, w in want.items():
            have = len(mm[l])
            print(f"  {l}: want {w/1e9:.2f}B  have {have/1e9:.2f}B  "
                f"epochs {w/have:.2f}")
        rng = np.random.default_rng(0)
        order = np.concatenate([np.full(want[l] // block, i)
                                for i, (l, _) in enumerate(ratio)])
        rng.shuffle(order)
        langs = [l for l, _ in ratio]
        pos = {l: 0 for l in langs}
        with open(f"{d}/train.bin", "wb") as fo:
            for i in order:
                l = langs[i]
                if pos[l] + block > len(mm[l]):
                    pos[l] = 0                      # wrap = next epoch
                mm[l][pos[l]:pos[l] + block].tofile(fo)
                pos[l] += block
        print(f"  -> {d}/train.bin  {os.path.getsize(d+'/train.bin')/1e9:.1f} GB")

    def mix_val(tokname, ratio=(("en",.40),("de",.35),("te",.25)), total_tokens=20_000_000):
        d = f"{ROOT}/bin/{tokname}"
        with open(f"{d}/val.bin", "wb") as fo:
            for l, p in ratio:
                a = np.memmap(f"{d}/{l}_val.bin", dtype=np.uint16, mode="r")
                n = min(int(total_tokens * p), len(a))
                a[:n].tofile(fo)
        print(f"  {tokname}/val.bin  {os.path.getsize(d+'/val.bin')/1e6:.0f} MB")

In [9]:
for tokname in ("cl100k_pat", "o200k_pat"):
    mix_val(tokname, total_tokens=15_000_000)

  cl100k_pat/val.bin  30 MB
  o200k_pat/val.bin  30 MB


In [10]:
import numpy as np, os
for t in ("cl100k_pat", "o200k_pat"):
    a = np.memmap(f"{ROOT}/bin/{t}/val.bin", dtype=np.uint16, mode="r")
    print(t, f"{len(a)/1e6:.1f}M tokens")

cl100k_pat 15.0M tokens
o200k_pat 15.0M tokens


In [12]:
import pickle
for t in ("cl100k_pat", "o200k_pat"):
    with open(f"E:/Project_SLM/bin/{t}/meta.pkl", "rb") as f:
        print(t, pickle.load(f))

cl100k_pat {'vocab_size': 48064}
o200k_pat {'vocab_size': 48064}


In [13]:
import shutil, os
for t, name in (("cl100k_pat","slm_cl100k"), ("o200k_pat","slm_o200k")):
    d = f"nanoGPT/data/{name}"       # adjust path to where you cloned it
    os.makedirs(d, exist_ok=True)
    for f in ("train.bin", "val.bin", "meta.pkl"):
        shutil.copy(f"E:/Project_SLM/bin/{t}/{f}", f"{d}/{f}")
    print(name, os.listdir(d))

slm_cl100k ['meta.pkl', 'train.bin', 'val.bin']
slm_o200k ['meta.pkl', 'train.bin', 'val.bin']


In [11]:
import pickle, os
for tokname in ("cl100k_pat", "o200k_pat"):
    d = f"E:/Project_SLM/bin/{tokname}"
    with open(f"{d}/meta.pkl", "wb") as f:
        pickle.dump({"vocab_size": 48064}, f)     # 48001 padded to a multiple of 64

In [ ]:
# for tokname in ("cl100k_pat", "o200k_pat"):
#     for lang in ("te", "de", "en"):
#         tokenize(tokname, lang, split="heldout")

  cl100k_pat/te: 5,774 docs  10.0M tokens  0.02 GB  3s
  cl100k_pat/de: 18,331 docs  11.2M tokens  0.02 GB  3s
  cl100k_pat/en: 11,133 docs  11.4M tokens  0.02 GB  3s
  o200k_pat/te: 5,774 docs  4.8M tokens  0.01 GB  1s
  o200k_pat/de: 18,331 docs  11.5M tokens  0.02 GB  3s
  o200k_pat/en: 11,133 docs  11.6M tokens  0.02 GB  3s


In [ ]:
# for tokname in ("cl100k_pat", "o200k_pat"):
#     mix(tokname)

  en: want 1.20B  have 3.64B  epochs 0.33
  de: want 1.05B  have 3.01B  epochs 0.35
  te: want 0.75B  have 1.70B  epochs 0.44
  -> E:/Project_SLM/bin/cl100k_pat/train.bin  6.0 GB
  en: want 1.20B  have 3.71B  epochs 0.32
  de: want 1.05B  have 3.09B  epochs 0.34
  te: want 0.75B  have 0.81B  epochs 0.92
  -> E:/Project_SLM/bin/o200k_pat/train.bin  6.0 GB


In [ ]:

# if __name__ == "__main__":
#     # for tokname in ("cl100k_pat", "o200k_pat"):
#     #     print(f"\n{tokname}")
#     #     for lang in ("te", "de", "en"):
#     #         tokenize(tokname, lang)
#     #     mix(tokname)



cl100k_pat
    200,704 docs  347M tok  81s
    401,408 docs  667M tok  157s
    600,064 docs  974M tok  230s
    800,768 docs  1276M tok  303s
    1,001,472 docs  1495M tok  357s
    1,200,128 docs  1696M tok  407s
  cl100k_pat/te: 1,202,250 docs  1698.7M tokens  3.40 GB  407s
    200,704 docs  133M tok  37s
    401,408 docs  247M tok  71s
    600,064 docs  365M tok  106s
    800,768 docs  489M tok  142s
    1,001,472 docs  617M tok  180s
    1,200,128 docs  758M tok  220s
    1,400,832 docs  895M tok  257s
    1,601,536 docs  1017M tok  293s
    1,800,192 docs  1134M tok  327s
    2,000,896 docs  1261M tok  364s
    2,201,600 docs  1394M tok  401s
    2,400,256 docs  1540M tok  441s
    2,600,960 docs  1669M tok  479s
    2,801,664 docs  1788M tok  514s
    3,000,320 docs  1915M tok  550s
    3,201,024 docs  2054M tok  588s
    3,401,728 docs  2197M tok  628s
    3,600,384 docs  2315M tok  663s
    3,801,088 docs  2431M tok  696s
    4,001,792 docs  2553M tok  732s
    4,200,448 docs